# dbt Testing

## Overview
dbt (data build tool) provides a structured testing framework built on top of the zero-row assertion pattern. Tests are declared in `schema.yml` and run with `dbt test`.

**dbt test types:**

| Type | Location | Defined in |
|---|---|---|
| **Generic (built-in)** | `schema.yml` | `tests:` block under a column |
| **Generic (custom)** | `macros/test_*.sql` | Jinja macro, reusable |
| **Singular** | `tests/*.sql` | Plain SQL file — one specific assertion |

**Running tests:**
```bash
dbt test                           # all tests
dbt test --select policies         # tests for one model
dbt test --select tag:critical     # tests by tag
dbt build                          # run models then tests together
dbt source freshness               # check source data recency
```

**Note:** These notebooks simulate dbt test logic with plain SQLite — no dbt installation required. All YAML and macro patterns shown are valid dbt syntax.

---

In [1]:
import sqlite3, json
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Mixed dataset: insurance + healthcare + financial transactions
CREATE TABLE customers (
    customer_id  TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    email        TEXT,
    province     TEXT NOT NULL,
    segment      TEXT NOT NULL,   -- 'retail','commercial','group'
    created_at   TEXT NOT NULL
);

CREATE TABLE policies (
    policy_id    TEXT PRIMARY KEY,
    customer_id  TEXT NOT NULL REFERENCES customers(customer_id),
    product_line TEXT NOT NULL,   -- 'auto','home','life','health'
    premium      REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'active','lapsed','cancelled'
    start_date   TEXT NOT NULL,
    end_date     TEXT
);

CREATE TABLE claims (
    claim_id     TEXT PRIMARY KEY,
    policy_id    TEXT NOT NULL REFERENCES policies(policy_id),
    claim_date   TEXT NOT NULL,
    amount       REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'open','approved','denied','paid'
    approved_by  TEXT
);

CREATE TABLE lab_results (
    result_id    TEXT PRIMARY KEY,
    patient_id   TEXT NOT NULL,
    test_name    TEXT NOT NULL,
    result_value REAL,
    unit         TEXT,
    result_date  TEXT NOT NULL,
    flagged      INTEGER DEFAULT 0
);

-- Intentionally seeded with some data quality issues for demos
""")

# Clean customers
customers = [
    ('C001','Alice Nguyen',    'alice@email.com',  'ON','retail',    '2022-01-15'),
    ('C002','Bob Harrington',  'bob@email.com',    'BC','commercial','2022-03-01'),
    ('C003','Fatima Al-Rashid','fatima@email.com', 'QC','group',     '2022-06-10'),
    ('C004','James MacLeod',   'james@email.com',  'NS','retail',    '2023-01-20'),
    ('C005','Mei-Ling Chen',   'mei@email.com',    'AB','commercial','2023-04-05'),
    ('C006','David Park',      None,               'ON','retail',    '2024-01-01'),  # null email
    ('C007','Sandra Okafor',   'sandra@email.com', 'ON','retail',    '2024-02-15'),
]
conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", customers)

# Policies — including some with data issues
policies = [
    ('POL-001','C001','auto',  1200.0,'active',   '2023-01-15','2024-01-15'),
    ('POL-002','C001','home',  1800.0,'active',   '2023-03-01','2024-03-01'),
    ('POL-003','C002','auto',  2400.0,'active',   '2022-06-01','2023-06-01'),
    ('POL-004','C003','life',   600.0,'lapsed',   '2022-09-01','2023-09-01'),
    ('POL-005','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),
    ('POL-006','C005','health',4800.0,'active',   '2023-05-01','2024-05-01'),
    ('POL-007','C006','home',  2100.0,'cancelled','2024-01-15','2024-06-15'),
    ('POL-008','C007','auto',  1100.0,'active',   '2024-03-01','2025-03-01'),
    ('POL-009','C002','home',  -500.0,'active',   '2024-01-01','2025-01-01'),  # negative premium (data issue)
    ('POL-010','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),  # duplicate of POL-005
]
conn.executemany("INSERT INTO policies VALUES (?,?,?,?,?,?,?)", policies)

# Claims
claims = [
    ('CLM-001','POL-001','2023-06-15',2200.0,'paid',    'Dr. Lee'),
    ('CLM-002','POL-003','2023-08-01', 500.0,'approved','Dr. Pham'),
    ('CLM-003','POL-005','2023-11-20',8500.0,'paid',    'Dr. Singh'),
    ('CLM-004','POL-006','2024-01-10', 350.0,'open',    None),
    ('CLM-005','POL-001','2024-02-28',1100.0,'denied',  'Dr. Lee'),
    ('CLM-006','POL-008','2024-04-15', 750.0,'approved','Dr. Pham'),
    ('CLM-007','POL-003','2024-05-01',99999.0,'open',   None),  # suspiciously large
]
conn.executemany("INSERT INTO claims VALUES (?,?,?,?,?,?)", claims)

# Lab results — with some quality issues
labs = [
    ('LAB-001','P001','HbA1c',    7.2,'%',        '2023-10-01',1),
    ('LAB-002','P001','eGFR',    68.0,'mL/min',   '2023-10-01',0),
    ('LAB-003','P002','HbA1c',    8.4,'%',        '2024-01-10',1),
    ('LAB-004','P002','Creatinine',115.0,'umol/L','2024-01-10',1),
    ('LAB-005','P003','HbA1c',    None,'%',        '2024-02-15',0),  # null value
    ('LAB-006','P001','HbA1c',    7.2,'%',        '2023-10-01',1),  # duplicate of LAB-001
    ('LAB-007','P004','eGFR',   -5.0,'mL/min',   '2024-03-01',0),  # impossible negative
    ('LAB-008','P003','HbA1c',    9.1,'%',        '2024-02-15',1),
]
conn.executemany("INSERT INTO lab_results VALUES (?,?,?,?,?,?,?)", labs)
conn.commit()

print("Testing dataset loaded:")
for tbl in ['customers','policies','claims','lab_results']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<15s}: {n} rows")
print()
print("Known data quality issues seeded:")
issues = [
    ("customers",    "C006 has NULL email"),
    ("policies",     "POL-009 has negative premium (-500)"),
    ("policies",     "POL-010 is a near-duplicate of POL-005"),
    ("claims",       "CLM-007 has suspiciously large amount (99999)"),
    ("lab_results",  "LAB-005 has NULL result_value"),
    ("lab_results",  "LAB-006 is an exact duplicate of LAB-001"),
    ("lab_results",  "LAB-007 has impossible negative eGFR (-5)"),
]
for tbl, issue in issues:
    print(f"  [{tbl}] {issue}")


Testing dataset loaded:
  customers      : 7 rows
  policies       : 10 rows
  claims         : 7 rows
  lab_results    : 8 rows

Known data quality issues seeded:
  [customers] C006 has NULL email
  [policies] POL-009 has negative premium (-500)
  [policies] POL-010 is a near-duplicate of POL-005
  [claims] CLM-007 has suspiciously large amount (99999)
  [lab_results] LAB-005 has NULL result_value
  [lab_results] LAB-006 is an exact duplicate of LAB-001
  [lab_results] LAB-007 has impossible negative eGFR (-5)


---
## Built-in generic tests (schema.yml)

In [2]:
print("=== dbt built-in generic tests ===")
print()

print("dbt's four built-in generic tests:")
dbt_tests = [
    ("unique",           "No duplicate values in the column",
     "- column: customer_id\n  tests:\n    - unique"),
    ("not_null",         "No NULL values in the column",
     "- column: premium\n  tests:\n    - not_null"),
    ("accepted_values",  "Every value is in a predefined list",
     "- column: status\n  tests:\n    - accepted_values:\n        values: ['active','lapsed','cancelled']"),
    ("relationships",    "Every FK value exists in the referenced table",
     "- column: customer_id\n  tests:\n    - relationships:\n        to: ref('dim_customer')\n        field: customer_id"),
]
for name, desc, yaml in dbt_tests:
    print(f"  {name}:")
    print(f"    Purpose: {desc}")
    print(f"    YAML:")
    for line in yaml.split('\n'):
        print(f"      {line}")
    print()

print("Full schema.yml for the policies model:")
schema_yml = """
version: 2

models:
  - name: policies
    description: "Insurance policies — one row per policy issued"
    columns:
      - name: policy_id
        description: "Unique policy identifier"
        tests:
          - unique
          - not_null

      - name: customer_id
        description: "FK to customers table"
        tests:
          - not_null
          - relationships:
              to: ref('customers')
              field: customer_id

      - name: product_line
        description: "Type of insurance product"
        tests:
          - not_null
          - accepted_values:
              values: ['auto', 'home', 'life', 'health']

      - name: premium
        description: "Annual premium amount (must be positive)"
        tests:
          - not_null

      - name: status
        description: "Current policy status"
        tests:
          - not_null
          - accepted_values:
              values: ['active', 'lapsed', 'cancelled']
"""
print(schema_yml)


=== dbt built-in generic tests ===

dbt's four built-in generic tests:
  unique:
    Purpose: No duplicate values in the column
    YAML:
      - column: customer_id
        tests:
          - unique

  not_null:
    Purpose: No NULL values in the column
    YAML:
      - column: premium
        tests:
          - not_null

  accepted_values:
    Purpose: Every value is in a predefined list
    YAML:
      - column: status
        tests:
          - accepted_values:
              values: ['active','lapsed','cancelled']

  relationships:
    Purpose: Every FK value exists in the referenced table
    YAML:
      - column: customer_id
        tests:
          - relationships:
              to: ref('dim_customer')
              field: customer_id

Full schema.yml for the policies model:

version: 2

models:
  - name: policies
    description: "Insurance policies — one row per policy issued"
    columns:
      - name: policy_id
        description: "Unique policy identifier"
        tests:


---
## Custom singular and generic tests

In [3]:
print("=== dbt custom generic and singular tests ===")
print()

print("Singular tests: one-off SQL queries in tests/ folder")
print("  Filename: tests/assert_no_negative_premiums.sql")
print("""
  -- tests/assert_no_negative_premiums.sql
  -- Returns rows on FAILURE (non-zero count = test fails)
  SELECT policy_id, premium
  FROM   {{ ref('policies') }}
  WHERE  premium < 0
""")

print("  Filename: tests/assert_lab_results_no_duplicates.sql")
print("""
  -- tests/assert_lab_results_no_duplicates.sql
  SELECT patient_id, test_name, result_date, COUNT(*) AS n
  FROM   {{ ref('lab_results') }}
  GROUP  BY patient_id, test_name, result_date
  HAVING COUNT(*) > 1
""")

print()
print("Custom generic tests (reusable across models):")
print("  Filename: macros/test_not_negative.sql")
print("""
  -- macros/test_not_negative.sql
  {% test not_negative(model, column_name) %}
  SELECT {{ column_name }}, COUNT(*) AS n_violations
  FROM   {{ model }}
  WHERE  {{ column_name }} < 0
  GROUP  BY {{ column_name }}
  {% endtest %}

  -- Usage in schema.yml:
  - name: premium
    tests:
      - not_negative
""")

print("  Filename: macros/test_accepted_range.sql")
print("""
  -- macros/test_accepted_range.sql
  {% test accepted_range(model, column_name, min_value, max_value) %}
  SELECT {{ column_name }}
  FROM   {{ model }}
  WHERE  {{ column_name }} IS NOT NULL
    AND  ({{ column_name }} < {{ min_value }}
       OR {{ column_name }} > {{ max_value }})
  {% endtest %}

  -- Usage in schema.yml:
  - name: result_value
    tests:
      - accepted_range:
          min_value: 0
          max_value: 30      # HbA1c physiologically impossible above 30
""")

print()
print("dbt test configuration options:")
configs = [
    ("severity: warn",        "Log a warning instead of failing the build"),
    ("severity: error",       "Fail the build (default)"),
    ("limit: 10",             "Show only first 10 failing rows in output"),
    ("store_failures: true",  "Save failing rows to a table in the DWH for review"),
    ("where: \"status='active'\"","Only test rows matching a WHERE clause (partial test)"),
    ("tags: ['critical']",    "Tag tests to run selectively: dbt test --select tag:critical"),
]
for cfg, desc in configs:
    print(f"  {cfg:<30s}  {desc}")


=== dbt custom generic and singular tests ===

Singular tests: one-off SQL queries in tests/ folder
  Filename: tests/assert_no_negative_premiums.sql

  -- tests/assert_no_negative_premiums.sql
  -- Returns rows on FAILURE (non-zero count = test fails)
  SELECT policy_id, premium
  FROM   {{ ref('policies') }}
  WHERE  premium < 0

  Filename: tests/assert_lab_results_no_duplicates.sql

  -- tests/assert_lab_results_no_duplicates.sql
  SELECT patient_id, test_name, result_date, COUNT(*) AS n
  FROM   {{ ref('lab_results') }}
  GROUP  BY patient_id, test_name, result_date
  HAVING COUNT(*) > 1


Custom generic tests (reusable across models):
  Filename: macros/test_not_negative.sql

  -- macros/test_not_negative.sql
  {% test not_negative(model, column_name) %}
  SELECT {{ column_name }}, COUNT(*) AS n_violations
  FROM   {{ model }}
  WHERE  {{ column_name }} < 0
  GROUP  BY {{ column_name }}
  {% endtest %}

  -- Usage in schema.yml:
  - name: premium
    tests:
      - not_negative



---
## dbt test commands and simulated run

In [4]:
print("=== dbt test commands and workflow ===")
print()

commands = [
    ("dbt test",
     "Run all tests in the project"),
    ("dbt test --select policies",
     "Run tests only for the policies model"),
    ("dbt test --select tag:critical",
     "Run only tests tagged 'critical'"),
    ("dbt test --select test_type:singular",
     "Run only singular (custom SQL) tests"),
    ("dbt test --select test_type:generic",
     "Run only generic (schema.yml) tests"),
    ("dbt build",
     "Run models + tests together (models first, then their tests)"),
    ("dbt test --store-failures",
     "Save failing rows to _dbt_test_failures schema"),
    ("dbt test --fail-fast",
     "Stop after first test failure"),
    ("dbt test --select source:raw_insurance",
     "Test source freshness and assertions"),
]
print("dbt CLI commands:")
print(f"  {'Command':<52s}  Effect")
print("  " + "-"*80)
for cmd, desc in commands:
    print(f"  {cmd:<52s}  {desc}")

print()
print("Source freshness tests (sources.yml):")
print("""
  version: 2
  sources:
    - name: raw_insurance
      database: analytics
      schema: raw
      freshness:
        warn_after: {count: 12, period: hour}
        error_after: {count: 24, period: hour}
      tables:
        - name: policies
          loaded_at_field: created_at     # column to check for freshness
          description: "Raw policy data from source system"

  # Run freshness check:
  # dbt source freshness
""")

print("Simulating dbt test output (our SQLite dataset):")
# Run the same tests dbt would run
tests = [
    ("unique.policies.policy_id",
     "SELECT policy_id, COUNT(*) FROM policies GROUP BY policy_id HAVING COUNT(*)>1"),
    ("not_null.policies.premium",
     "SELECT policy_id FROM policies WHERE premium IS NULL"),
    ("accepted_values.policies.status",
     "SELECT policy_id, status FROM policies WHERE status NOT IN ('active','lapsed','cancelled')"),
    ("not_null.policies.customer_id",
     "SELECT policy_id FROM policies WHERE customer_id IS NULL"),
    ("relationships.policies.customer_id",
     "SELECT p.policy_id FROM policies p LEFT JOIN customers c ON c.customer_id=p.customer_id WHERE c.customer_id IS NULL"),
    ("singular.assert_no_negative_premiums",
     "SELECT policy_id, premium FROM policies WHERE premium < 0"),
    ("singular.assert_no_duplicate_labs",
     "SELECT patient_id,test_name,result_date,COUNT(*) n FROM lab_results GROUP BY patient_id,test_name,result_date HAVING COUNT(*)>1"),
]
passed = failed = 0
for name, sql in tests:
    rows = conn.execute(sql).fetchall()
    if rows:
        print(f"  FAIL  {name}  ({len(rows)} records)")
        failed += 1
    else:
        print(f"  pass  {name}")
        passed += 1
print(f"\n  {passed} passed, {failed} failed")


=== dbt test commands and workflow ===

dbt CLI commands:
  Command                                               Effect
  --------------------------------------------------------------------------------
  dbt test                                              Run all tests in the project
  dbt test --select policies                            Run tests only for the policies model
  dbt test --select tag:critical                        Run only tests tagged 'critical'
  dbt test --select test_type:singular                  Run only singular (custom SQL) tests
  dbt test --select test_type:generic                   Run only generic (schema.yml) tests
  dbt build                                             Run models + tests together (models first, then their tests)
  dbt test --store-failures                             Save failing rows to _dbt_test_failures schema
  dbt test --fail-fast                                  Stop after first test failure
  dbt test --select source:raw_insura

---
## Common Pitfalls

**1. Only using built-in tests (unique, not_null) and skipping business rules**
Built-in tests check structural integrity. But `premium > 0`, `end_date > start_date`, and `claim_amount < coverage_amount` are business rules that require singular or custom generic tests. A model that passes all four built-in tests can still have completely wrong data.

**2. `accepted_values` list not maintained as source system adds new values**
`accepted_values: ['auto','home','life']` will fail every time the product team adds a new product line. Either maintain the list in a seed file and join to it, or use a relationships test against a dimension table that owns the valid values.

**3. Using `severity: warn` for everything to keep the build green**
Warnings are easy to ignore. Reserve `severity: warn` for non-critical checks (e.g., optional fields that are sometimes NULL by design). For data that feeds production dashboards or regulatory reports, use the default `severity: error` so violations break the build and demand attention.

**4. Forgetting `store_failures: true` for complex failure debugging**
When a test fails, dbt shows a summary count. Without `store_failures: true`, you must re-run the test SQL manually to see the actual bad rows. Enable storage for critical tests so the failure details are always available in the `_dbt_test_failures` schema.

**5. Not testing source tables (only tested models)**
Untested source tables are the most common entry point for bad data. Add `sources.yml` with freshness checks and `not_null`/`unique` tests on source PKs. Run `dbt source freshness` before every build.

**6. Singular tests with no `{{ ref() }}` — hardcoded table names**
`SELECT * FROM policies WHERE premium < 0` hardcodes the table. `SELECT * FROM {{ ref('policies') }} WHERE premium < 0` respects dbt's target schema and environment (dev vs prod). Always use `ref()` in singular tests.


---
*sql_methods_library - Samantha McGarrigle*